# A/B Test 01 — Click-Through Rate (CTR)

**Experiment.** A new homepage banner (`treatment`) is tested against the old one (`control`).

**Question.** Did the new banner achieve a higher click-through rate, and is the difference statistically significant?

**Metric type.** `clicked` is a binary 0/1 outcome, so CTR is a **proportion** → the right test is a **two-proportion z-test**.


In [1]:
import pandas as pd
from statsmodels.stats.proportion import proportions_ztest

df = pd.read_parquet("../data/ab_test_data.parquet")
df.head()

,user_id,group,clicked,purchased,order_value,returned_30d
0,T01163,treatment,1,0,0.00,1
1,T04385,treatment,0,0,0.00,1
2,C01902,control,1,1,96.31,0
3,C03397,control,0,0,0.00,0
4,T05695,treatment,1,0,0.00,1


In [2]:
# Data quality checks (run before trusting any metric)
print("shape:", df.shape)
print("\nnulls:\n", df.isnull().sum())
print("\nduplicate user_id:", df["user_id"].duplicated().sum())

# SRM (sample ratio mismatch): groups should be ~50/50
print("\ngroup sizes:\n", df["group"].value_counts())

# Binary columns must contain only 0/1
for col in ["clicked", "purchased", "returned_30d"]:
    print(col, "unique:", sorted(df[col].unique()))

shape: (12000, 6)

nulls:
 user_id         0
group           0
clicked         0
purchased       0
order_value     0
returned_30d    0
dtype: int64

duplicate user_id: 0

group sizes:
 group
treatment    6000
control      6000
Name: count, dtype: int64
clicked unique: [np.int64(0), np.int64(1)]
purchased unique: [np.int64(0), np.int64(1)]
returned_30d unique: [np.int64(0), np.int64(1)]


## Compute the metric

CTR = clicks / users, computed per group.

In [3]:
summary = df.groupby("group")["clicked"].agg(["sum", "count"])
summary["ctr"] = summary["sum"] / summary["count"]
print(summary)

            sum  count     ctr
group                         
control    1755   6000  0.2925
treatment  2220   6000  0.3700


## Statistical test — two-proportion z-test

The metric is a rate (proportion), so we compare the two proportions with a z-test.

In [4]:
clicks = summary["sum"].values     # [control_clicks, treatment_clicks]
n      = summary["count"].values   # [control_total,  treatment_total]
stat, pval = proportions_ztest(clicks, n)

print(f"control CTR = {summary['ctr']['control']:.4f} | treatment CTR = {summary['ctr']['treatment']:.4f}")
print(f"p-value = {pval:.4f}")
print("Significant" if pval < 0.05 else "Not significant")

control CTR = 0.2925 | treatment CTR = 0.3700
p-value = 0.0000
Significant


## Result

CTR rose from **0.2925 (control)** to **0.3700 (treatment)**, p ≈ 0.0000 → statistically significant.

- **Absolute lift:** +7.75 percentage points (0.3700 − 0.2925)
- **Relative lift:** ≈ +26% (7.75 / 29.25)

Report the absolute change first (it is concrete and honest); mention the relative lift too, since relative numbers can look exaggerated on a small baseline.

## Concepts

### Why a z-test (not a t-test)?
`clicked` is binary 0/1, so the metric is a **proportion**. Comparing proportions → z-test. For a continuous numeric metric (e.g. order value) we would compare **means** → t-test.

### What the p-value means
The p-value is the probability of observing a difference at least this large **if there were truly no difference between the groups**. Here it is below 0.05, so the difference is very unlikely to be chance alone.
*Note:* p < 0.05 does not "prove" the effect — it quantifies how surprising the data is under the no-difference assumption.

### Why check group sizes (SRM)?
Balanced groups (6000 / 6000) indicate the randomization worked. A large imbalance would signal a **sample ratio mismatch (SRM)** — a sign the assignment is broken, and the comparison can't be trusted.

### Is a higher CTR enough to ship?
No. A click is not the goal. CTR's key **downstream** metric is **conversion** — clicks only matter if they turn into purchases (otherwise it's clickbait). We would also check **guardrail** metrics (bounce rate, page load speed, complaints, returns) to ensure nothing degraded.
